# Perform an inventory audit of an Excel HTA model

This is attempting to recreate the first stage of the Microsoft CoPilot prompt i.e. identifying all inputs. This includes inputs that are hardcoded into formulas.

The diagram below illustrates the general approach (subject to change):

<img src='images/model_qc_workflow.png' width='800' height='800' />





## 1. Imports

### 1.1. General imports

In [2]:
import sys
from pathlib import Path
from rich import print as printr
import json
import pandas as pd

import warnings
# to ignore a openpyxl UserWarning
warnings.filterwarnings('ignore')  # Suppress all warnings

### 1.2 `xl-ray`

In [3]:
from xl_ray.extractor import (
    load_workbooks, 
    extract_metadata, 
    extract_vba_modules, 
    extract_named_ranges,
    extract_worksheets 
)

from xl_ray.schema import ExcelModelData

from xl_ray.audit import (
    detect_magic_numbers, # TM note - I wonder if this is useful for LLM hint?
    add_cell_context,
    add_context_to_parameters,
    extract_numeric_constants,
)

### 1.3 LLM workflow imports

In [23]:
# standard langchain 
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# custom in input_selection module
from input_selection import (
    SELECTION_INSTRUCTIONS,
    INPUT_CLASSIFICATION_INSTRUCTIONS,
    SheetSelection,
    InputClassification,
    build_selection_payload,
    selected_sheet_names,
    validate_selection,
    build_selection_text,
    build_input_classification_payload
)

## 2. Import model via `xl-ray`

In [5]:
TEST_MODEL = "models/Exeter_model_template_Mar25.xlsm"

In [6]:
test_file = Path(TEST_MODEL)

if not test_file.exists():
    print(f"Error: Could not find {test_file}")
    sys.exit(1)

print(f"Loading '{test_file.name}'...")
wb_formulas, wb_values = load_workbooks(test_file)

print("Extracting VBA modules...")
vba_modules = extract_vba_modules(test_file, wb_formulas)

print("Extracting Named Ranges...")
named_ranges = extract_named_ranges(wb_formulas)

print("Extracting Worksheets, Cells, and Tables (this may take a moment)...")
worksheets = extract_worksheets(wb_formulas, wb_values)

metadata = extract_metadata(
    path=Path(test_file),
    wb_formulas=wb_formulas, 
    vba_modules=vba_modules, 
    named_ranges=named_ranges
)

metadata.has_data_tables = any(ws.tables for ws in worksheets.values())
metadata.has_array_formulas = any(
    cell.is_array_formula for ws in worksheets.values() for cell in ws.cells.values()
)

excel_model = ExcelModelData(
    metadata=metadata,
    worksheets=worksheets,
    named_ranges=named_ranges,
    vba_modules=vba_modules
)

print("Import of Excel model complete.")

Loading 'Exeter_model_template_Mar25.xlsm'...
Extracting VBA modules...
Extracting Named Ranges...
Extracting Worksheets, Cells, and Tables (this may take a moment)...
Import of Excel model complete.


## 3. LLM functions

In [7]:
def select_worksheets_using_LLM(
    payload: str,
    system_prompt: str,
    base_url: str = "http://127.0.0.1:8001/v1",
    selected_model: str = "google/gemma-4-26B-A4B-it",
    random_seed: int | None = None,
    temperature: float = 1.0,
) -> SheetSelection:
    """
    Select worksheets for detailed input identification.

    `prompt` should contain the compact workbook summary.

    Returns a validated SheetSelection, not an AIMessage.
    """
    model = ChatOpenAI(
        base_url=base_url,
        api_key="EMPTY",
        model=selected_model,
        temperature=temperature,
        **({"seed": random_seed} if random_seed is not None else {}),
    )

    # used to get structured output
    structured_model = model.with_structured_output(
        SheetSelection,
        method="json_schema",
        include_raw=False,
    )

    selection = structured_model.invoke(
        [
            SystemMessage(content=system_prompt.strip()),
            HumanMessage(content=payload),
        ]
    )

    return selection

In [8]:
def classify_inputs_using_LLM(
    payload: str,
    system_prompt: str,
    base_url: str = "http://127.0.0.1:8001/v1",
    selected_model: str = "google/gemma-4-26B-A4B-it",
    random_seed: int | None = None,
    temperature: float = 1.0,
) -> InputClassification:
    """
    Classify candidate input cells from one worksheet.

    Parameters
    ----------
    payload : str
        JSON payload containing candidates and workbook context.
    system_prompt : str
        Input-classification instructions.
    base_url : str, optional
        OpenAI-compatible server URL.
    selected_model : str, optional
        Model identifier served by the endpoint.
    random_seed : int or None, optional
        Seed passed to the server. Default is None.
    temperature : float, optional
        Sampling temperature. Default is 1.0.

    Returns
    -------
    InputClassification
        Parsed and schema-validated classification results.
    """
    model = ChatOpenAI(
        base_url=base_url,
        api_key="EMPTY",
        model=selected_model,
        temperature=temperature,
        **({"seed": random_seed} if random_seed is not None else {}),
    )

    structured_model = model.with_structured_output(
        InputClassification,
        method="json_schema",
        include_raw=False,
    )

    classification = structured_model.invoke(
        [
            SystemMessage(content=system_prompt.strip()),
            HumanMessage(content=payload),
        ]
    )

    return classification

In [32]:
def classify_selected_sheets(
    excel_model: ExcelModelData,
    sheets: list[str],
    *,
    system_prompt: str = INPUT_CLASSIFICATION_INSTRUCTIONS,
    rows_above: int = 3,
    rows_below: int = 1,
    columns_left: int = 3,
    columns_right: int = 1,
    base_url: str = "http://127.0.0.1:8001/v1",
    selected_model: str = "google/gemma-4-26B-A4B-it",
    random_seed: int | None = None,
    temperature: float = 1.0,
    verbose: bool = True,
) -> dict[str, InputClassification]:
    """
    Prepare and classify selected worksheets, collecting their results.

    Parameters
    ----------
    excel_model : ExcelModelData
        Extracted workbook.
    sheets : list[str]
        Worksheet names to analyse, in processing order.
    system_prompt : str, optional
        Input-classification instructions.
    rows_above, rows_below, columns_left, columns_right : int, optional
        Context window sizes. Defaults are 3, 1, 3 and 1 respectively.
    base_url : str, optional
        OpenAI-compatible server URL.
    selected_model : str, optional
        Model identifier served by the endpoint.
    random_seed : int or None, optional
        Seed passed to the server. Default is None.
    temperature : float, optional
        Sampling temperature. Default is 1.0.
    verbose : bool, optional
        Print progress messages. Default is True.

    Returns
    -------
    dict[str, InputClassification]
        Classification results keyed by worksheet name.

    Raises
    ------
    ValueError
        If a requested worksheet does not exist.

    Notes
    -----
    Calls ``classify_inputs_using_LLM`` separately for each populated
    candidate list. Duplicate sheet names are processed once.
    """
    sheets = list(dict.fromkeys(sheets))

    missing = [
        sheet for sheet in sheets
        if sheet not in excel_model.worksheets
    ]
    if missing:
        raise ValueError(f"Worksheets not found: {missing}")

    if not sheets:
        return {}

    workbook_summary = build_selection_payload(excel_model)
    results = {}

    for index, sheet_name in enumerate(sheets, start=1):
        parameters = extract_numeric_constants(
            excel_model,
            sheets=sheet_name,
        )

        if verbose:
            print(
                f"[{index}/{len(sheets)}] {sheet_name}: "
                f"{len(parameters)} numeric candidates"
            )

        if not parameters:
            results[sheet_name] = InputClassification(
                target_sheet=sheet_name,
                candidates=[],
                limitations=["No numeric candidates found."],
            )
            continue

        parameters_with_context = add_context_to_parameters(
            excel_model,
            parameters,
            rows_above=rows_above,
            rows_below=rows_below,
            columns_left=columns_left,
            columns_right=columns_right,
        )

        payload = build_input_classification_payload(
            model_data=excel_model,
            sheet_name=sheet_name,
            candidates=parameters_with_context,
            workbook_summary=workbook_summary,
        )

        payload_text = json.dumps(
            payload,
            ensure_ascii=False,
            allow_nan=False,
            default=json_default,
        )

        # The existing single-sheet LLM function stays separate.
        results[sheet_name] = classify_inputs_using_LLM(
            payload=payload_text,
            system_prompt=system_prompt,
            base_url=base_url,
            selected_model=selected_model,
            random_seed=random_seed,
            temperature=temperature,
        )

    return results

## 4. Script to select sheets to include in audit

### 4.1. Build the worksheet selection prompt

In [9]:
print("Building prompt")
payload = build_selection_payload(
    excel_model,
    max_rows_per_sheet = 12,
    max_cells_per_row = 10,
)

# convert JSON to plain text to reduce charactor count
workbook_summary_as_text= build_selection_text(payload)

print("Workbook summary characters:", len(workbook_summary_as_text))

Building prompt
Workbook summary characters: 71907


### 4.2 Prompt LLM to select worksheets

In [10]:
# These are the LLMs currently running.
ENDPOINTS = {
    "qwen": {
        "model": "Qwen/Qwen3.5-35B-A3B",
        "url": "http://127.0.0.1:8000/v1"
    },
    "gemma4": {
        "model": "google/gemma-4-26B-A4B-it", 
        "url": "http://127.0.0.1:8001/v1"
    },
    # If using isambard
    "deepseek":{
        "model": "deepseek-ai/DeepSeek-V4.1-Flash",
        "url": "http://localhost:11436/v1" 
    },
}

In [11]:
%%time

selected_endpoint = "qwen"

print(
    f"{ENDPOINTS[selected_endpoint]["model"]} is selecting sheets "
    "to include in audit"
)

# run the prompt
selection = select_worksheets_using_LLM(
    payload=workbook_summary_as_text,
    system_prompt = SELECTION_INSTRUCTIONS,
    base_url=ENDPOINTS[selected_endpoint]["url"],
    selected_model=ENDPOINTS[selected_endpoint]["model"],
    random_seed=42,
)

Qwen/Qwen3.5-35B-A3B is selecting sheets to include in audit
CPU times: user 340 ms, sys: 21.8 ms, total: 362 ms
Wall time: 23.3 s


In [12]:
type(selection)

input_selection.SheetSelection

### 4.3. Save results and convert to `DataFrame` 

In [13]:
# Validate worksheet names and coverage.
selection = validate_selection(
    selection.model_dump_json(),
    excel_model,
)

sheets_to_audit = selected_sheet_names(
    selection,
    include_review=True,
)

# Convert the validated Pydantic objects to a DataFrame.
selection_df = pd.DataFrame(
    [
        {
            "sheet": item.sheet,
            "decision": item.decision,
            "role": item.likely_role,
            "reason": item.reason,
            "evidence": "\n".join(item.evidence),
            "uncertainty": item.uncertainty,
        }
        for item in selection.sheets
    ],
    columns=[
        "sheet",
        "decision",
        "role",
        "reason",
        "evidence",
        "uncertainty",
    ],
)
 
selection_df["selected_for_audit"] = selection_df["sheet"].isin(
    sheets_to_audit
)

# Retain the worksheet objects for the next LLM stage.
selected_worksheets = {
    name: excel_model.worksheets[name]
    for name in sheets_to_audit
}

In [14]:
# show sheets excluded from audit
selection_df[selection_df["selected_for_audit"] == False]

,sheet,decision,role,reason,evidence,uncertainty,selected_for_audit
0,Cover,skip,unknown,"Worksheet contains metadata only (client name,...",Row 5: C5='[Client name]'\nRow 10: C10='PenTAG...,,False
1,Background,skip,unknown,Worksheet contains descriptive text and strate...,Row 6: C6='Model diagram '\nRow 34: C34='Cells...,,False
18,Model QC,skip,unknown,Contains checklist for quality control of the ...,Row 1: A1='White box tests' | B1='Test no.' | ...,,False
19,Navigation,skip,unknown,Contains navigation links and layout notes only.,Row 1: C1='Navigation'\nRow 10: E10='Backgroun...,,False
24,References,skip,unknown,Contains bibliographic citations and sources f...,Row 4: E4=1 | F4='PSSRU 2023' | G4='PSSRU 2023...,,False


In [15]:
# show sheets included in stage 2
selection_df[selection_df["selected_for_audit"] == True]

,sheet,decision,role,reason,evidence,uncertainty,selected_for_audit
2,Model_settings,inspect,inputs,Contains user-specified model settings includi...,Row 18: C18='Life Years discount rate' | D18=0...,,True
3,Treatment_costs,inspect,inputs,Contains treatment cost inputs including unit ...,Row 7: C7=2 | D7='Treatment 1' | E7=100 | G7=2...,,True
4,HCRU,inspect,inputs,Contains health care resource use unit costs a...,Row 7: C7=2 | D7='Resource 2' | E7=20 | F7=201...,,True
5,Societal_costs,inspect,inputs,Contains societal resource use unit costs and ...,Row 16: C16=2 | D16='Well' | E16='Presenteeism...,,True
6,Inflation,inspect,inputs,Contains historical inflation rates by year so...,Row 5: D5='2008/2009' | E5=2009 | G5=0.027 | H...,,True
7,Effectiveness,inspect,inputs,Contains treatment effectiveness inputs includ...,Row 11: D11='Intervention 2 vs No treatment' |...,,True
8,LifeTable,review,inputs,Large mortality table containing life expectan...,Row 87: C87=73 | F87=[formula] =1-EXP(-1*(-LN(...,Sampled rows are heavily formula-driven; uncer...,True
9,Adverse_events,inspect,inputs,"Contains adverse event costs, HRQoL decrements...",Row 5: C5=1 | D5='Event 1' | E5=100 | G5=[form...,,True
10,HRQoL,inspect,inputs,"Contains utility values, regression parameters...",Row 8: D8='Baseline utility' | E8=0.8\nRow 14:...,,True
11,Parameters,inspect,inputs,Centralized parameter table linking all inputs...,Row 13: D13='Model_settings' | G13='pop_settin...,,True


In [16]:
# Take from the copilot oneshot example. 
# Unique list of sheets that contain flagged inputs.
copilot_included = [
    "Model_settings",
    "Treatment_costs",
    "HCRU",
    "Societal_costs",
    "Effectiveness",
    "Adverse_events",
    "HRQoL",
    "LifeTable",
    "Parameters",
    "PSA",
    "OWSA",
    "Scenario_analysis",
    "Engine(1)",
    "Engine(2)",
    "Engine(3)",
]

len(copilot_included)

15

## 5. What about xl-ray magic numbers?

I don't need to use the LLM for finding all cells with hard coded values when I have `xl-ray`

The function `detect_magic_numbers` captures all formulas with hard coded values.  Returned as a list

**To consider:**

* Do I just need to organise these and use the LLM to provide the context and understanding?
* What about just cells that are hard coded?
* Can `xl-ray` provide simple functionality to help the LLM on where parameters might be stored?
* How hard would it be to identifier potential ICER cells with `xl-ray`

**Maybe it doesn't matter at this stage?**



In [17]:
from IPython.display import display, Markdown

# returns a list
cells_with_hard_coded_settings = detect_magic_numbers(excel_model)

# peek
display(Markdown("**Preview of magic numbers**\n"))
print(f"No. magic numbers in formulas: {len(cells_with_hard_coded_settings)}")
cells_with_hard_coded_settings[:2]

**Preview of magic numbers**


No. magic numbers in formulas: 3562


[{'sheet': 'Model_settings',
  'cell': 'D16',
  'formula': '=ctrl_setting_CycleLength*7/365.25',
  'magic_numbers_found': ['7', '365.25']},
 {'sheet': 'HCRU',
  'cell': 'H15',
  'formula': '=(INDEX(p_r_HCRU_HealthStateResourceUse,C15)/365.25)*ctrl_setting_CycleLength*7',
  'magic_numbers_found': ['365.25', '7']}]

In [18]:
# include the sheets already screened in by the LLM 
sheets_to_include= list(
    selection_df[selection_df["selected_for_audit"] == True]['sheet']
)
sheets_to_include[:2]

['Model_settings', 'Treatment_costs']

In [19]:
# get the cell references containing numeric constants
# FOR SHEET 0 only while testing...
parameters = extract_numeric_constants(
    excel_model, 
    sheets=sheets_to_include[0]
)

print(len(parameters))

# add context cells 3 above, 1 below, 3 to the left, 1 to the right.
parameter_with_context = add_context_to_parameters(
    excel_model,
    parameters,
    rows_above=3,
    rows_below=1,
    columns_left=3,
    columns_right=1,
)

# example of what this looks like.
parameter_with_context[0]

12


{'sheet': 'Model_settings',
 'cell': 'D6',
 'value': 31,
 'visibility': 'Visible',
 'context': [{'cell': 'C4',
   'value': 'Baseline population',
   'formula': None,
   'row_offset': -2,
   'column_offset': -1},
  {'cell': 'C5',
   'value': 'Parameter',
   'formula': None,
   'row_offset': -1,
   'column_offset': -1},
  {'cell': 'D5',
   'value': 'Value',
   'formula': None,
   'row_offset': -1,
   'column_offset': 0},
  {'cell': 'E5',
   'value': 'SD',
   'formula': None,
   'row_offset': -1,
   'column_offset': 1},
  {'cell': 'C6',
   'value': 'Age (male)',
   'formula': None,
   'row_offset': 0,
   'column_offset': -1},
  {'cell': 'E6',
   'value': 12,
   'formula': None,
   'row_offset': 0,
   'column_offset': 1},
  {'cell': 'C7',
   'value': 'Age (female)',
   'formula': None,
   'row_offset': 1,
   'column_offset': -1},
  {'cell': 'D7',
   'value': 31,
   'formula': None,
   'row_offset': 1,
   'column_offset': 0},
  {'cell': 'E7',
   'value': 12,
   'formula': None,
   'row_offs

## Classify inputs for one sheet

In [21]:
import json
from datetime import date, datetime, time

def json_default(value):
    """Handle date/time values in neighbouring context cells."""
    if isinstance(value, (date, datetime, time)):
        return value.isoformat()
    raise TypeError(f"Cannot serialise {type(value).__name__}")

In [24]:
# Reuse workbook_summary here if you already created it.
workbook_summary = build_selection_payload(excel_model)

# Build the payload for the sheet used in your extraction step.
classification_payload = build_input_classification_payload(
    model_data=excel_model,
    sheet_name=sheets_to_include[0],
    candidates=parameter_with_context,
    workbook_summary=workbook_summary,
)

# Convert the payload into the text sent to the LLM.
payload_text = json.dumps(
    classification_payload,
    ensure_ascii=False,
    allow_nan=False,
    default=json_default,
)

In [26]:
%%time

selected_endpoint = "qwen"

print(
    f"{ENDPOINTS[selected_endpoint]["model"]} is classifying inputs "
    "for human reviewer"
)

# Call the LLM for this sheet only.
classification = classify_inputs_using_LLM(
    payload=payload_text,
    system_prompt=INPUT_CLASSIFICATION_INSTRUCTIONS,
    base_url=ENDPOINTS[selected_endpoint]["url"],
    selected_model=ENDPOINTS[selected_endpoint]["model"],
    random_seed=42,
    temperature=1.0,
)

Qwen/Qwen3.5-35B-A3B is classifying inputsfor human reviewer
CPU times: user 16.4 ms, sys: 3.79 ms, total: 20.2 ms
Wall time: 10.8 s


In [27]:
import pandas as pd

# Convert the classification records into rows.
classification_df = pd.DataFrame(
    [candidate.model_dump() for candidate in classification.candidates],
    columns=[
        "sheet", "cell", "decision", "parameter_name", "unit",
        "input_type", "category", "named_range", "evidence", "notes",
    ],
)

# Recover the original values rather than asking the LLM to repeat them.
values_df = pd.DataFrame(
    [
        {
            "sheet": parameter["sheet"],
            "cell": parameter["cell"],
            "value": parameter["value"],
        }
        for parameter in parameter_with_context
    ],
    columns=["sheet", "cell", "value"],
)

review_df = classification_df.merge(
    values_df,
    on=["sheet", "cell"],
    how="left",
    validate="one_to_one",
)

# Make the evidence list easier to read in a notebook table.
review_df["evidence"] = review_df["evidence"].apply(
    lambda items: "\n".join(items)
)

# Arrange the columns for human review.
review_df = review_df[
    [
        "decision",
        "parameter_name",
        "value",
        "unit",
        "input_type",
        "category",
        "sheet",
        "cell",
        "named_range",
        "evidence",
        "notes",
    ]
]

review_df

,decision,parameter_name,value,unit,input_type,category,sheet,cell,named_range,evidence,notes
0,include,Age (male),31.000,years,Clinical Input,Clinical,Model_settings,D6,pop_setting_baseline_ageM,Context shows 'Baseline population' section wi...,"Patient baseline characteristic, likely used f..."
1,include,Age SD (male),12.000,years,Clinical Input,Clinical,Model_settings,E6,NaN,Context shows 'SD' column header in E5\nValue ...,Standard deviation parameter associated with a...
2,include,Age (female),31.000,years,Clinical Input,Clinical,Model_settings,D7,pop_setting_baseline_ageF,Context shows 'Baseline population' section wi...,"Patient baseline characteristic, likely used f..."
3,include,Age SD (female),12.000,years,Clinical Input,Clinical,Model_settings,E7,NaN,Context shows 'SD' column header in E5\nValue ...,Standard deviation parameter associated with a...
4,include,%female,0.520,proportion,Clinical Input,Clinical,Model_settings,D8,pop_setting_baseline_female,Context shows '%female' label in C8\nValue 0.5...,Population gender distribution parameter.
5,include,Weight (KGs),80.000,KGs,Clinical Input,Clinical,Model_settings,D9,pop_setting_baseline_weight,Context shows 'Weight (KGs)' label in C9\nValu...,Patient baseline characteristic.
6,include,Willingness-to-pay threshold (£ per QALY),20000.000,GBP,Cost,Other,Model_settings,D14,ctrl_setting_WTP,Context shows 'Willingness-to-pay threshold (£...,Economic threshold used for cost-effectiveness...
7,include,Cycle length (weeks),4.000,weeks,Assumption,Other,Model_settings,D15,ctrl_setting_CycleLength,Context shows 'Cycle length (weeks)' label in ...,Model structure parameter defining cycle durat...
8,include,Costs discount rate,0.035,decimal (annual),Discount Rate,Other,Model_settings,D17,ctrl_setting_DR_Costs,Context shows 'Costs discount rate' label in C...,Annual discount rate for costs.
9,include,Life Years discount rate,0.035,decimal (annual),Discount Rate,Other,Model_settings,D18,ctrl_setting_DR_LYs,Context shows 'Life Years discount rate' label...,Annual discount rate for life years.


In [30]:
def classifications_to_dataframe(
    results: dict[str, InputClassification],
    excel_model: ExcelModelData,
) -> pd.DataFrame:
    """
    Convert worksheet classifications into a combined review table.

    Parameters
    ----------
    results : dict[str, InputClassification]
        Results returned by ``classify_sheets_using_LLM``.
    excel_model : ExcelModelData
        Original extracted workbook supplying the numeric values.

    Returns
    -------
    pandas.DataFrame
        Candidate decisions, original values, evidence and notes.

    Notes
    -----
    Preserves worksheet and candidate order. Worksheet-level limitations
    remain available in ``results`` rather than being repeated per row.
    """
    columns = [
        "decision",
        "parameter_name",
        "value",
        "unit",
        "input_type",
        "category",
        "sheet",
        "cell",
        "named_range",
        "evidence",
        "notes",
    ]

    rows = []

    for classification in results.values():
        for candidate in classification.candidates:
            row = candidate.model_dump()

            row["value"] = (
                excel_model.worksheets[candidate.sheet]
                .cells[candidate.cell]
                .value
            )
            row["evidence"] = "\n".join(candidate.evidence)

            rows.append(row)

    return pd.DataFrame(rows, columns=columns)

In [35]:
%%time

selected_endpoint = "qwen"

print(
    f"{ENDPOINTS[selected_endpoint]["model"]} is classifying inputs "
    "for human reviewer"
)

results = classify_selected_sheets(
    excel_model,
    sheets=sheets_to_include[:2],
    base_url=ENDPOINTS[selected_endpoint]["url"],
    selected_model=ENDPOINTS[selected_endpoint]["model"],
    random_seed=42,
    temperature=1.0,
)

Qwen/Qwen3.5-35B-A3B is classifying inputs for human reviewer
[1/2] Model_settings: 12 numeric candidates
[2/2] Treatment_costs: 109 numeric candidates
CPU times: user 198 ms, sys: 14.9 ms, total: 213 ms
Wall time: 1min 39s


In [36]:
review_df = classifications_to_dataframe(results, excel_model)
review_df

,decision,parameter_name,value,unit,input_type,category,sheet,cell,named_range,evidence,notes
0,include,Age (male),31.00,years,Clinical Input,Clinical,Model_settings,D6,pop_setting_baseline_ageM,Context shows 'Baseline population' section wi...,"Patient baseline characteristic, likely used f..."
1,include,Age SD (male),12.00,years,Clinical Input,Clinical,Model_settings,E6,NaN,Context shows 'SD' column header in E5\nValue ...,Standard deviation parameter associated with a...
2,include,Age (female),31.00,years,Clinical Input,Clinical,Model_settings,D7,pop_setting_baseline_ageF,Context shows 'Baseline population' section wi...,"Patient baseline characteristic, likely used f..."
3,include,Age SD (female),12.00,years,Clinical Input,Clinical,Model_settings,E7,NaN,Context shows 'SD' column header in E5\nValue ...,Standard deviation parameter associated with a...
4,include,%female,0.52,proportion,Clinical Input,Clinical,Model_settings,D8,pop_setting_baseline_female,Context shows '%female' label in C8\nValue 0.5...,Population gender distribution parameter.
...,...,...,...,...,...,...,...,...,...,...,...
116,exclude,Combined Costs Index,3.00,NaN,NaN,NaN,Treatment_costs,C67,NaN,Index column for combined costs\nSequential nu...,Exclude: Sequential index used for identificat...
117,exclude,Combined Costs Index,4.00,NaN,NaN,NaN,Treatment_costs,C68,NaN,Index column for combined costs\nSequential nu...,Exclude: Sequential index used for identificat...
118,exclude,Combined Costs Index,5.00,NaN,NaN,NaN,Treatment_costs,C69,NaN,Index column for combined costs\nSequential nu...,Exclude: Sequential index used for identificat...
119,exclude,Combined Costs Index,6.00,NaN,NaN,NaN,Treatment_costs,C70,NaN,Index column for combined costs\nSequential nu...,Exclude: Sequential index used for identificat...


In [40]:
len(review_df[review_df['decision'] == "include"])

55

In [41]:
review_df[review_df['decision'] == "include"]

,decision,parameter_name,value,unit,input_type,category,sheet,cell,named_range,evidence,notes
0,include,Age (male),31.000,years,Clinical Input,Clinical,Model_settings,D6,pop_setting_baseline_ageM,Context shows 'Baseline population' section wi...,"Patient baseline characteristic, likely used f..."
1,include,Age SD (male),12.000,years,Clinical Input,Clinical,Model_settings,E6,NaN,Context shows 'SD' column header in E5\nValue ...,Standard deviation parameter associated with a...
2,include,Age (female),31.000,years,Clinical Input,Clinical,Model_settings,D7,pop_setting_baseline_ageF,Context shows 'Baseline population' section wi...,"Patient baseline characteristic, likely used f..."
3,include,Age SD (female),12.000,years,Clinical Input,Clinical,Model_settings,E7,NaN,Context shows 'SD' column header in E5\nValue ...,Standard deviation parameter associated with a...
4,include,%female,0.520,proportion,Clinical Input,Clinical,Model_settings,D8,pop_setting_baseline_female,Context shows '%female' label in C8\nValue 0.5...,Population gender distribution parameter.
5,include,Weight (KGs),80.000,KGs,Clinical Input,Clinical,Model_settings,D9,pop_setting_baseline_weight,Context shows 'Weight (KGs)' label in C9\nValu...,Patient baseline characteristic.
6,include,Willingness-to-pay threshold (£ per QALY),20000.000,GBP,Cost,Other,Model_settings,D14,ctrl_setting_WTP,Context shows 'Willingness-to-pay threshold (£...,Economic threshold used for cost-effectiveness...
7,include,Cycle length (weeks),4.000,weeks,Assumption,Other,Model_settings,D15,ctrl_setting_CycleLength,Context shows 'Cycle length (weeks)' label in ...,Model structure parameter defining cycle durat...
8,include,Costs discount rate,0.035,decimal (annual),Discount Rate,Other,Model_settings,D17,ctrl_setting_DR_Costs,Context shows 'Costs discount rate' label in C...,Annual discount rate for costs.
9,include,Life Years discount rate,0.035,decimal (annual),Discount Rate,Other,Model_settings,D18,ctrl_setting_DR_LYs,Context shows 'Life Years discount rate' label...,Annual discount rate for life years.
